# 03 – Splitting protocols
Sets aside the 2024–2025 holdout, defines the walk-forward and random splits, checks how badly the random split breaks time order, and runs a first logistic regression comparison.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, matthews_corrcoef

from google.colab import drive
drive.mount('/content/drive')

BASE     = "/content/drive/MyDrive/CS5500_dissertation"
FEATURES = f"{BASE}/features"
RESULTS  = f"{BASE}/results"

TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]
SEED    = 42

np.random.seed(SEED)
print("Ready")

## 2. Load features and separate the final holdout
Everything from 2024 onward is set aside and not touched by any protocol comparison.

In [ ]:
DEV_END      = "2023-12-31"
HOLDOUT_FROM = "2024-01-01"

dev, holdout = {}, {}

for t in TICKERS:
    f = pd.read_csv(f"{FEATURES}/{t}_features.csv", index_col=0, parse_dates=True)
    dev[t]     = f.loc[:DEV_END]
    holdout[t] = f.loc[HOLDOUT_FROM:]
    print(f"{t}: dev {len(dev[t])} rows, holdout {len(holdout[t])} rows")

PREDICTORS = [c for c in dev["AAPL"].columns
              if not c.startswith("target") and c != "price_today"]

print(f"\n{len(PREDICTORS)} predictors")
print(f"Development period ends {DEV_END}; holdout from {HOLDOUT_FROM}")

## 3. The two splitters
Walk-forward: 1,000-row initial window, 250-row test folds, step of 250. Random: a 20% test split.

In [ ]:
INITIAL_TRAIN = 1000
TEST_SIZE     = 250
STEP          = 250


def expanding_walk_forward(n_rows, initial_train=INITIAL_TRAIN,
                           test_size=TEST_SIZE, step=STEP):
    """Yield (train_positions, test_positions) with training always strictly
    before testing. Positions are integer row indices into a date-sorted frame."""
    start = initial_train

    while start + test_size <= n_rows:
        train = np.arange(0, start)
        test  = np.arange(start, start + test_size)
        yield train, test
        start += step


def random_split(n_rows, test_frac=0.2, seed=SEED):
    """Shuffle all rows and hold out a random fraction.

    This is the leakage-prone protocol. It is included because it is what
    much of the applied literature does, not because it is defensible."""
    rng = np.random.default_rng(seed)
    positions = rng.permutation(n_rows)

    n_test = int(n_rows * test_frac)
    return positions[n_test:], positions[:n_test]


n = len(dev["AAPL"])
folds = list(expanding_walk_forward(n))

print(f"AAPL development rows: {n}")
print(f"Walk-forward folds:    {len(folds)}\n")

idx = dev["AAPL"].index
for i, (tr, te) in enumerate(folds, 1):
    print(f"  fold {i:>2}: train {len(tr):>5} rows to {idx[tr[-1]].date()} | "
          f"test {idx[te[0]].date()} to {idx[te[-1]].date()}")

## 4. Verify the splitters behave as claimed
Checks that walk-forward never trains on the future, and measures how much the random split does.

In [ ]:
print("CHECK A - walk-forward: no training row may postdate any test row\n")

violations = 0
for i, (tr, te) in enumerate(folds, 1):
    last_train  = idx[tr].max()
    first_test  = idx[te].min()
    ok = last_train < first_test
    if not ok:
        violations += 1
    verdict = "ok" if ok else "VIOLATION"
    print(f"  fold {i:>2}: last train {last_train.date()} < "
          f"first test {first_test.date()}  ->  {verdict}")

print(f"\nViolations: {violations}")
print("PASS - walk-forward never trains on the future" if violations == 0
      else "FAIL - splitter is broken")

In [ ]:
print("CHECK B - random split: how badly does it violate temporal order?\n")

tr_rand, te_rand = random_split(n)

earliest_test = idx[te_rand].min()
n_after = (idx[tr_rand] > earliest_test).sum()

print(f"  Training rows:                     {len(tr_rand)}")
print(f"  Test rows:                         {len(te_rand)}")
print(f"  Earliest test date:                {earliest_test.date()}")
print(f"  Training rows AFTER that date:     {n_after} "
      f"({n_after / len(tr_rand):.1%} of training data)")
print(f"  Latest training date:              {idx[tr_rand].max().date()}")
print(f"  Latest test date:                  {idx[te_rand].max().date()}")

print("\n  Interpretation: the model is trained on data from the future")
print("  relative to most of what it is then asked to predict.")

## 5. Scaling: fold-fitted versus globally fitted
A scaler stores the mean and standard deviation of each feature.

In [ ]:
X_all = dev["AAPL"][PREDICTORS].values

tr0, te0 = folds[0]

global_scaler = StandardScaler().fit(X_all)          # sees everything
fold_scaler   = StandardScaler().fit(X_all[tr0])     # sees only fold 1 training

comparison = pd.DataFrame({
    "feature":     PREDICTORS,
    "mean_global": global_scaler.mean_,
    "mean_fold":   fold_scaler.mean_,
    "std_global":  np.sqrt(global_scaler.var_),
    "std_fold":    np.sqrt(fold_scaler.var_),
})

comparison["std_pct_diff"] = (
    100 * (comparison.std_global - comparison.std_fold) / comparison.std_fold
)

out = comparison.reindex(
    comparison.std_pct_diff.abs().sort_values(ascending=False).index
)

print("Scaler statistics, global fit vs fold-1 training fit (AAPL)")
print("std_pct_diff shows how much the global standard deviation differs\n")
print(out[["feature", "std_fold", "std_global", "std_pct_diff"]]
      .round(4).to_string(index=False))

print(f"\nLargest divergence: {out.std_pct_diff.abs().max():.1f}%")
print("Every non-zero value is future information reaching the training stage.")

## 6. First inflation estimate
Logistic regression under two protocols, on the same data with the same target.

In [ ]:
def run_leakage_prone(df):
    """Random split, scaler fitted on the full dataset before splitting."""
    X = df[PREDICTORS].values
    y = df["target_direction"].values

    # The leak: statistics computed across train and test together
    X_scaled = StandardScaler().fit_transform(X)

    tr, te = random_split(len(df))

    model = LogisticRegression(max_iter=1000, random_state=SEED)
    model.fit(X_scaled[tr], y[tr])
    pred = model.predict(X_scaled[te])

    return y[te], pred


def run_sound(df):
    """Expanding walk-forward, scaler refitted on each training fold only."""
    X = df[PREDICTORS].values
    y = df["target_direction"].values

    y_true, y_pred = [], []

    for tr, te in expanding_walk_forward(len(df)):
        scaler = StandardScaler().fit(X[tr])     # training fold only

        model = LogisticRegression(max_iter=1000, random_state=SEED)
        model.fit(scaler.transform(X[tr]), y[tr])

        y_pred.extend(model.predict(scaler.transform(X[te])))
        y_true.extend(y[te])

    return np.array(y_true), np.array(y_pred)


rows = []

for t in TICKERS:
    df = dev[t]

    for label, fn in [("leakage_prone", run_leakage_prone), ("sound", run_sound)]:
        y_true, y_pred = fn(df)

        acc = accuracy_score(y_true, y_pred)
        mcc = matthews_corrcoef(y_true, y_pred)
        se  = np.sqrt(acc * (1 - acc) / len(y_true))

        rows.append({
            "ticker":   t,
            "protocol": label,
            "n":        len(y_true),
            "accuracy": round(acc, 4),
            "ci_low":   round(acc - 1.96 * se, 4),
            "ci_high":  round(acc + 1.96 * se, 4),
            "mcc":      round(mcc, 4),
        })

protocol_results = pd.DataFrame(rows)
print(protocol_results.to_string(index=False))

In [ ]:
pivot = protocol_results.pivot(index="ticker", columns="protocol",
                               values=["accuracy", "mcc"])

pivot[("accuracy", "inflation_pts")] = (
    100 * (pivot[("accuracy", "leakage_prone")] - pivot[("accuracy", "sound")])
).round(2)

print("Protocol comparison - logistic regression, direction target\n")
print(pivot.round(4).to_string())

mean_inflation = pivot[("accuracy", "inflation_pts")].mean()
print(f"\nMean accuracy inflation across five assets: {mean_inflation:.2f} percentage points")

protocol_results.to_csv(f"{RESULTS}/protocol_comparison_logreg.csv", index=False)
print(f"Saved to {RESULTS}/protocol_comparison_logreg.csv")

## 7. Append to the results log

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append(f"## {today} - Notebook 03: splitting protocols")
lines.append("")
lines.append("**Design**")
lines.append(f"- Development period: start to {DEV_END}")
lines.append(f"- Final holdout: {HOLDOUT_FROM} onward, untouched")
lines.append(f"- Walk-forward: initial train {INITIAL_TRAIN}, test fold {TEST_SIZE}, step {STEP}")
lines.append(f"- Folds per asset: {len(folds)}")
lines.append("")
lines.append("**Splitter verification**")
lines.append(f"- Walk-forward temporal ordering: {violations} violations across all folds")
lines.append(f"- Random split: {n_after} training rows ({n_after/len(tr_rand):.1%}) "
             f"postdate the earliest test row")
lines.append("")
lines.append("**Scaler divergence (AAPL, fold 1)**")
lines.append(f"- Largest difference between globally fitted and fold-fitted "
             f"standard deviation: {out.std_pct_diff.abs().max():.1f}%")
lines.append("")
lines.append("**Protocol comparison, logistic regression on direction target**")
lines.append("")
lines.append(protocol_results.to_string(index=False))
lines.append("")
lines.append(f"- Mean accuracy inflation: {mean_inflation:.2f} percentage points")
lines.append("- Only the split strategy and scaler fitting differ between arms; "
             "model, features and target are identical.")
lines.append("")
lines.append("**Next:** Notebook 04 - target formulation (price level vs return) "
             "and the full single-factor ablation")

entry = "\n".join(lines)

with open(f"{BASE}/results_log.md", "a") as f:
    f.write(entry)

print(entry)